In [ ]:
import backtrader as bt
import datetime
import pandas as pd


In [ ]:
# from mongofeed import MongoData

# td = datetime.datetime.combine(datetime.date.today(),datetime.time())
# feed = MongoData(
#     db='stock_etf',
#     dataname='etf_159915',
#     fromdate=datetime.datetime(2011,12,9),
#     #fromdate=datetime.datetime(2015,12,9),
#     todate=td
# )
# csv data source
dataframe = pd.read_csv('./csv_files/000300.XSHG.csv', parse_dates=True, index_col=0)

# pandasdata feeder
feed = bt.feeds.PandasData(dataname=dataframe, openinterest=None)

In [ ]:
MaT=['SMA','EMA','WMA','DEMA','TEMA','TRIMA','KAMA','MAMA','T3']

# Create a Stratey
class CyStrategy(bt.Strategy):
    params = (
        ('period', 5),
    )
    
    def log(self, txt):
        ''' Logging function for this strategy'''
        dt = self.datas[0].datetime.date(0)
        print('%s, %s' % (dt.isoformat(), txt))
        
    def __init__(self):
        
        self.kama = bt.indicators.KAMA(
            self.dnames.cy_weekly,
            period=self.p.period,
        )
        #self.kama.plotinfo.plotmaster = self.dnames.cy_daily
        
        self.dmi = bt.indicators.DM(
            self.dnames.cy_daily,
            period=self.p.period,
        )
        
        self.kd = bt.indicators.StochasticFull(
            self.dnames.cy_daily,
            period=self.p.period,
#             period_dfast=self.p.slowk,
#             period_dslow=self.p.slowd,
        )
    
        #self.cross = bt.indicators.CrossOver(self.macd.macd, self.macd.macdsignal, plot=False)
        #self.above = bt.And(self.macd.macd>0.0, self.macd.macdsignal>0.0)
        #self.buy_signal = bt.And(self.above, self.cross==1)
        
        self.cross = bt.indicators.CrossOver(self.kd.percD, self.kd.percDSlow, plot=False)
        
        self.buy_signal = self.cross==1
        self.sell_signal = self.cross==-1
        
        # ex indicators
        #self.mfi = bt.talib.MFI(self.data0.high, self.data0.low, self.data0.close, self.data0.volume)
        
        # cycle
        #self.ht_dc = bt.talib.HT_DCPERIOD(self.data0)
        
        # internal variables
        # To keep track of pending orders
        self.order = None
        self.start_len = 0
        
    def notify_order(self, order):
        if order.status in [order.Submitted, order.Accepted]:
            # Buy/Sell order submitted/accepted to/by broker - Nothing to do
            return

        # Check if an order has been completed
        # Attention: broker could reject order if not enough cash
#         if order.status == order.Completed:
#             if order.isbuy():
#                 self.log(
#                     'BUY EXECUTED, Size: %d, Price: %.2f, Cost: %.2f, Comm %.2f' %
#                     (order.executed.size,
#                      order.executed.price,
#                      order.executed.value,
#                      order.executed.comm))

#             else:  # Sell
#                 self.log('SELL EXECUTED, Size: %d, Price: %.2f, Cost: %.2f, Comm %.2f' %
#                          (order.executed.size,
#                           order.executed.price,
#                           order.executed.value,
#                           order.executed.comm))

#         elif order.status in [order.Canceled, order.Margin, order.Rejected]:
#             self.log('Order Canceled/Margin/Rejected')

        # Write down: no pending order
        self.order = None
    
    def notify_trade(self, trade):
        if not trade.isclosed:
            return

#         self.log('OPERATION PROFIT, GROSS %.2f, NET %.2f' %
#                  (trade.pnl, trade.pnlcomm))

    def prenext(self):
        #self.log('Prenext called')
        pass
    
    def nextstart(self):
        self.start_len = len(self)
        #self.log('Start Len:{}'.format(self.start_len))
        
    def next(self):
        #self.log('d0 {:f}'.format(self.data0.close[0]))
        #self.log('dname {}'.format(self.data0._name))
        # Check if an order is pending ... if yes, we cannot send a 2nd one
        if self.order:
            return
        
        # Check if we are in the market
        if not self.position:
            # Not yet ... we MIGHT BUY if ...
            if self.buy_signal[0]:
                # BUY, BUY, BUY!!! (with all possible default parameters)
                #self.log('BUY CREATE')
                # Keep track of the created order to avoid a 2nd order
                self.order = self.buy()
        else:
            # Already in the market ... we might sell
            if self.sell_signal[0]:
                # SELL, SELL, SELL!!! (with all possible default parameters)
                #self.log('SELL CREATE')

                # Keep track of the created order to avoid a 2nd order
                self.order = self.sell()
    def stop(self):
        #self.log('P:{:2d} MA:{} EV:{:.2f}'.format(self.p.period, MaT[self.p.sigmatype], self.broker.getvalue()))
        self.valid_len = len(self) - self.start_len
        self.log('Valid trading len:{}'.format(self.valid_len))

In [ ]:
cerebro = bt.Cerebro(oldtrades=True, oldbuysell=True)
#feed.plotinfo.plot = False
cerebro.adddata(feed, name='cy_daily')
cy_weekly = cerebro.resampledata(feed, name='cy_weekly', timeframe=bt.TimeFrame.Weeks )
#cy_weekly.plotinfo.plot = False

cerebro.addstrategy(CyStrategy, period=9)
#cerebro.optstrategy(CyStrategy, period=range(2, 7), sigmatype=[bt.talib.MA_Type.KAMA, bt.talib.MA_Type.SMA,bt.talib.MA_Type.EMA])

#
cerebro.broker.setcash(100000.0)

# 手续费万5
cerebro.broker.setcommission(0.0005)

cerebro.broker.set_coc(True)
cerebro.broker.set_fundstartval(50)

# it only works when only one strategy is running
#cerebro.addwriter(bt.WriterFile, csv=True, out='wlog.csv')

cerebro.addsizer(bt.sizers.AllInSizerInt, percents=99)

print('Starting Portfolio Value: %.2f' % cerebro.broker.getvalue())

#cerebro.addanalyzer(bt.analyzers.AnnualReturn)
#cerebro.addanalyzer(bt.analyzers.PyFolio)
#cerebro.addanalyzer(bt.analyzers.TimeDrawDown)
#cerebro.addanalyzer(bt.analyzers.TradeAnalyzer)
cerebro.addanalyzer(bt.analyzers.SQN)
#cerebro.addanalyzer(bt.analyzers.VWR)
#cerebro.addanalyzer(bt.analyzers.SharpeRatio, riskfreerate=0.01)

result = cerebro.run()

print('Final Portfolio Value: %.2f' % cerebro.broker.getvalue())

In [ ]:
strat = result[0]
for a_name in strat.analyzers.getnames():
    strat.analyzers.getbyname(a_name).pprint()
    
print('valid len:{}, trades:{}'.format(strat.valid_len,strat.analyzers.sqn.get_analysis()['trades']))
print(strat.valid_len / strat.analyzers.sqn.get_analysis()['trades'])

In [ ]:
params = dict(
    style='candle',
    barup='#FF0033',
    bardown='#32CD32',
    volup='#F66269',
    voldown='#43A047',
)

cerebro.plot(
    iplot=True, 
    #numfigs=8,
    #start=datetime.date(2017,1,1),
    #end=datetime.date(2019,1,1),
    **params
)

In [ ]:
from backtrader_plotting import Bokeh
from bokeh.plotting import output_file

candle_params = dict(
    style='bar',
    barup='#FF0033',
    bardown='#32CD32',
    volup='#F66269',
    voldown='#43A047',
)

output_file('cy_research.html')
b = Bokeh(output_mode='save', **candle_params)
cerebro.plot(b)